# 04 — Backtest & Robustness Analysis

Evaluates the economic tradability of LOB-based signals via a decile-sorted long/short backtest under multiple transaction cost scenarios, plus regime-based robustness checks.

**Key question**: Does statistical predictability translate into economic profitability?

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.utils.paths import load_config
from src.data.load_data import load_processed
from src.validation.walk_forward import chronological_split
from src.models.train_model import train_and_evaluate
from src.backtest.strategy import run_backtest, run_backtest_grid
from src.backtest.performance import (
    plot_equity_curve, plot_cost_sensitivity, format_summary_table
)
from src.analysis.robustness import run_robustness_suite, plot_regime_ic

config = load_config()
sns.set_theme(style='whitegrid', font_scale=1.1)

## 1. Generate Predictions

In [ ]:
all_predictions = {}
all_tests = {}

for sym in config['assets']:
    df = load_processed(sym, 'features')
    train, val, test = chronological_split(df)
    all_tests[sym] = test
    all_predictions[sym] = {}
    
    for h in ['1s', '5s', '10s', '30s']:
        result = train_and_evaluate(train, test, f'y_return_{h}', 'ridge', 'all', 'regression')
        if 'predictions' in result:
            all_predictions[sym][h] = result['predictions']
            ic = result.get('pearson_ic', float('nan'))
            print(f"{sym} {h}: IC={ic:.4f}, {len(result['predictions'])} predictions")

## 2. Backtest Results

In [ ]:
for sym in config['assets']:
    test = all_tests[sym]
    predictions = all_predictions[sym]
    
    summary = run_backtest_grid(test, predictions, config=config)
    formatted = format_summary_table(summary)
    
    print(f"\n{'='*70}")
    print(f"  {sym} — Backtest Summary")
    print(f"{'='*70}")
    display(formatted)

## 3. Equity Curves

In [ ]:
for sym in config['assets']:
    test = all_tests[sym]
    predictions = all_predictions[sym]
    
    for h in ['1s', '5s', '10s', '30s']:
        if h in predictions:
            bt = run_backtest(test, predictions[h], horizon=h, config=config)
            fig = plot_equity_curve(bt, symbol=sym)
            plt.show()

## 4. Cost Sensitivity

In [ ]:
for sym in config['assets']:
    test = all_tests[sym]
    predictions = all_predictions[sym]
    summary = run_backtest_grid(test, predictions, config=config)
    
    fig = plot_cost_sensitivity(summary, metric='sharpe')
    plt.suptitle(f'{sym} — Sharpe Ratio by Cost Scenario', fontsize=14, fontweight='bold', y=1.02)
    plt.show()
    
    # Highlight the critical finding
    for h in ['1s', '5s', '10s', '30s']:
        row = summary[(summary['horizon'] == h) & (summary['cost_scenario'] == 'net_medium')]
        if not row.empty:
            sharpe = row['Sharpe'].values[0]
            status = '✓ VIABLE' if sharpe > 0 else '✗ NOT VIABLE'
            print(f"  {h}: Medium-cost Sharpe = {sharpe:.1f}  {status}")

## 5. Cross-Asset Comparison

In [ ]:
horizons = ['1s', '5s', '10s', '30s']
comparison = []
for sym in config['assets']:
    summary = run_backtest_grid(all_tests[sym], all_predictions[sym], config=config)
    for h in horizons:
        gross = summary[(summary['horizon'] == h) & (summary['cost_scenario'] == 'gross')]
        medium = summary[(summary['horizon'] == h) & (summary['cost_scenario'] == 'net_medium')]
        if not gross.empty and not medium.empty:
            comparison.append({
                'Asset': sym, 'Horizon': h,
                'Gross Sharpe': gross['Sharpe'].values[0],
                'Net Sharpe (medium)': medium['Sharpe'].values[0],
                'Hit Rate': gross['Hit %'].values[0],
                'Trades': gross['Trades'].values[0],
            })

comp_df = pd.DataFrame(comparison)
display(comp_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, metric in enumerate(['Gross Sharpe', 'Net Sharpe (medium)']):
    ax = axes[i]
    for sym in config['assets']:
        subset = comp_df[comp_df['Asset'] == sym]
        ax.plot(subset['Horizon'], subset[metric], 'o-', label=sym, linewidth=2, markersize=8)
    ax.set_title(metric, fontsize=13)
    ax.set_xlabel('Horizon')
    ax.set_ylabel('Sharpe Ratio')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.legend()

plt.suptitle('Gross vs Net Sharpe: The Transaction Cost Gap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Regime Analysis

In [ ]:
for sym in config['assets']:
    test = all_tests[sym]
    preds = all_predictions[sym].get('5s')
    if preds is None:
        continue
    
    regime_results = run_robustness_suite(test, preds, target='y_return_5s')
    
    print(f"\n{'='*60}")
    print(f"  {sym} — Regime IC (5s horizon)")
    print(f"{'='*60}")
    for name, rdf in regime_results.items():
        if not rdf.empty:
            print(f"\n{name}:")
            display(rdf.round(4))
    
    fig = plot_regime_ic(regime_results, title_suffix=f' — {sym} 5s')
    plt.show()

## 7. Key Takeaways

### Statistical Predictability
- Walk-forward IC of 0.16–0.35 across horizons and assets
- All IC t-statistics > 2.0 (statistically significant)
- Signal decays from 5s peak to 30s

### Economic Tradability
- **1s**: Gross Sharpe > 150 → Net Sharpe deeply negative. NOT tradable.
- **5s**: Marginal — breaks even under medium costs for ETH only.
- **10s**: ETH survives medium costs; BTC marginal.
- **30s**: Both assets profitable under all cost scenarios.

### The Fundamental Tension
Signal strength peaks at 1–5s, but tradability peaks at 30s. The signal's half-life (~10–15s) means the most predictable returns cannot be captured profitably. This is the classic microstructure research finding: **statistical alpha ≠ tradable alpha**.